# Knowledge Graphs and Semantic Technologies -- OWL tutorial

First install owlready2 if you don't already have it, and have a quick look at the [documentation](https://owlready2.readthedocs.io/en/v0.36/onto.html#).

In [1]:
## Uncomment if you do not have owlrl installed (you should have it installed from the RDFS tutorial)
#import sys
#!{sys.executable} -m pip install rdflib  owlready2 pandas

import pandas as pd
from rdflib import Graph, Literal, Namespace, RDF, URIRef, OWL
from rdflib.namespace import DC, FOAF

from owlready2 import *

Let's start loading some data from a .CSV file. We are going to create an ontology that describes the data inside.
We already did part of this using the semantics of RDF(S), now we'll use the semantics of [OWL](https://www.w3.org/TR/2012/REC-owl2-primer-20121211/) through owlready2. 

Remember that an ontology is often an application ontology, meaning that it is built with a specific task in mind. 
We could model _everything_ within a certain domain in the most ontologically correct way possible, _or_ **we could model the domain in accordance with the application's task.** 


**Your task and domain:** You are a broadcaster that has just digitised its radio archives into a digital music archive (DMA), and aims to play more interesting tracks by discovering their 'hidden treasures', by making unexpected and potentially interesting relations between tracks visible to the users (which are journalists and program makers).


**Exercise 1** 

1. look at the .csv files in the folder /data/musicoset_metadata/ and load them into pandas dataframes (use display.max_columns to show all columns). 
2. initialise an empty ontology using owlready2
3. using owlready2, create a hierarchy of classes and subclasses that describe the entities in your dataframes
4. using owrleady2, create properties and subproperties their properties, and how the classes relate to one another (using domain and range). If it helps: draw out your ontology in https://app.diagrams.net/
    - create: object properties, data properties, functional properties
5. using owlready2, add class restrictions
6. create invididuals of your classes, and provide them with attributes using your properties! 
7. write simple queries to retrieve your individuals following: https://owlready2.readthedocs.io/en/v0.36/onto.html#simple-queries. What kind of things would journalists and program makers like to retrieve? 
6. save your asserted owl file

In [2]:
#load each csv in a different df
df_albums = pd.read_csv('data/musicoset_metadata/albums.csv', sep='\t')
pd.set_option('display.max_columns', None)
df_albums.head()

,album_id,name,billboard,artists,popularity,total_tracks,album_type,image_url
0,5n1GSzC1Reao29ScnpLYqp,Dying To Live,Dying To Live,{'46SHBwWsqBkxI7EeeBEQG7': 'Kodak Black'},83,16,album,https://i.scdn.co/image/db2133234d458f432ca207...
1,6UYZEYjpN1DYRW0kqFy9ZE,Championships,Championships,{'20sxb77xiYeusSH8cVdatc': 'Meek Mill'},85,19,album,https://i.scdn.co/image/77eb7c17cafe5503c58661...
2,7uVimUILdzSZG4KKKWToq0,Christmas (Deluxe Special Edition),Christmas,{'1GxkXlMwML1oSg5eLPiAz3': 'Michael Bublé'},60,20,album,https://i.scdn.co/image/2d6ee8d4fb5a45abf35cd3...
3,35s58BRTGAEWztPo9WqCIs,Spider-Man: Into the Spider-Verse (Soundtrack ...,Spider-Man: Into The Spider-Verse,{'0LyfQWJT6nXafLPZqxe9Of': 'Various Artists'},92,13,compilation,https://i.scdn.co/image/3aa37254a41cf96e815725...
4,41GuZcammIkupMPKH2OJ6I,ASTROWORLD,ASTROWORLD,{'0Y5tJX1MQlPlqiwlOH1tJY': 'Travis Scott'},91,17,album,https://i.scdn.co/image/cdca7dc20c778ada42fb18...


In [3]:
df_artists = pd.read_csv("data/musicoset_metadata/artists.csv", sep='\t')
pd.set_option('display.max_columns', None)
df_artists.head()

,artist_id,name,followers,popularity,artist_type,main_genre,genres,image_url
0,66CXWjxzNUsdJxJ2JdwvnR,Ariana Grande,34554242.0,96,singer,dance pop,"['dance pop', 'pop', 'post-teen pop']",https://i.scdn.co/image/b1dfbe843b0b9f54ab2e58...
1,26VFTg2z8YR0cCuwLzESi2,Halsey,7368242.0,90,singer,dance pop,"['dance pop', 'electropop', 'etherpop', 'indie...",https://i.scdn.co/image/22a5f3d8c42bc7cb55215e...
2,0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,6313709.0,94,rapper,pop,"['pop', 'pop rap', 'rap']",https://i.scdn.co/image/dc5eba5e032c2e5bc4d42c...
3,246dkjvS1zLTtiykXe5h60,Post Malone,16737002.0,96,rapper,dfw rap,"['dfw rap', 'pop', 'rap']",https://i.scdn.co/image/f9d8b742b66609f12da023...
4,1zNqQNIdeOUZHb8zbZRFMX,Swae Lee,483032.0,89,singer,trap music,['trap music'],https://i.scdn.co/image/a177469870b41f7e17e3b5...


In [4]:
df_releases = pd.read_csv("data/musicoset_metadata/releases.csv", sep='\t')
pd.set_option('display.max_columns', None)
df_releases.head()

,artist_id,album_id,release_date,release_date_precision
0,46SHBwWsqBkxI7EeeBEQG7,5n1GSzC1Reao29ScnpLYqp,2018-12-14,day
1,20sxb77xiYeusSH8cVdatc,6UYZEYjpN1DYRW0kqFy9ZE,2018-11-30,day
2,1GxkXlMwML1oSg5eLPiAz3,7uVimUILdzSZG4KKKWToq0,2012-11-09,day
3,0LyfQWJT6nXafLPZqxe9Of,35s58BRTGAEWztPo9WqCIs,2018-12-14,day
4,0Y5tJX1MQlPlqiwlOH1tJY,41GuZcammIkupMPKH2OJ6I,2018-08-03,day


In [5]:
df_songs = pd.read_csv("data/musicoset_metadata/songs.csv", sep='\t')
pd.set_option('display.max_columns', None)
df_songs.head()

,song_id,song_name,billboard,artists,popularity,explicit,song_type
0,3e9HZxeyfWwjeyPAMmWSSQ,"thank u, next","('Thank U, Next', 'Ariana Grande')",{'66CXWjxzNUsdJxJ2JdwvnR': 'Ariana Grande'},86,True,Solo
1,5p7ujcrUXASCNwRaWNHR1C,Without Me,"('Without Me', 'Halsey')",{'26VFTg2z8YR0cCuwLzESi2': 'Halsey'},87,True,Solo
2,2xLMifQCjDGFmkHkpNLD9h,SICKO MODE,"('Sicko Mode', 'Travis Scott')",{'0Y5tJX1MQlPlqiwlOH1tJY': 'Travis Scott'},85,True,Solo
3,3KkXRkHbMCARz0aVfEt68P,Sunflower - Spider-Man: Into the Spider-Verse,('Sunflower (Spider-Man: Into The Spider-Verse...,"{'246dkjvS1zLTtiykXe5h60': 'Post Malone', '1zN...",92,False,Collaboration
4,1rqqCSm0Qe4I9rUvWncaom,High Hopes,"('High Hopes', 'Panic! At The Disco')",{'20JZFwl6HVl6yg8a4H3ZqK': 'Panic! At The Disco'},86,False,Solo


In [6]:
df_tracks = pd.read_csv("data/musicoset_metadata/tracks.csv", sep='\t')
pd.set_option('display.max_columns', None)
df_tracks.head()

,song_id,album_id,track_number,release_date,release_date_precision
0,3e9HZxeyfWwjeyPAMmWSSQ,2fYhqwDWXjbpjaIJPEfKFw,11,2019-02-08,day
1,5p7ujcrUXASCNwRaWNHR1C,0zzrCTzvL4ZmR42xF46Afm,1,2018-10-04,day
2,2xLMifQCjDGFmkHkpNLD9h,41GuZcammIkupMPKH2OJ6I,3,2018-08-03,day
3,3KkXRkHbMCARz0aVfEt68P,35s58BRTGAEWztPo9WqCIs,2,2018-12-14,day
4,1rqqCSm0Qe4I9rUvWncaom,6ApYSpXF8GxZAgBTHDzYge,4,2018-06-22,day


In [7]:
#initialize empty ontology
onto = get_ontology("http://example.org/music_ontology.owl")

In [8]:
#Create classes - each class is subclass of thing
with onto: 
    class Album(Thing): pass

    class Artist(Thing): pass

    class Song(Thing): pass

    class Person(Thing): pass

    class Genre(Thing): pass

    class SubGenre(Genre): pass

    class SoloArtist(Artist): pass

    class ArtistType(Artist): pass

In [9]:
print(SoloArtist.ancestors())
print(list(onto.classes()))

{owl.Thing, music_ontology.SoloArtist, music_ontology.Artist}
[music_ontology.Album, music_ontology.Artist, music_ontology.Song, music_ontology.Person, music_ontology.Genre, music_ontology.SubGenre, music_ontology.SoloArtist, music_ontology.ArtistType]


In [10]:
#Create properties and subproperties
with onto:
    class hasArtist(ObjectProperty):
        domain = [Album]
        range = [Artist]
    
    class hasArtistSong(ObjectProperty):
        domain = [Song]
        range  = [Artist]
    
    class createdBy(ObjectProperty):
        domain = [Song]
        range = [Artist]

    class writtenBy(createdBy): pass
        
    class isOfAlbum(Song >> Album): pass

    class isOfGenre(Song >> Genre): pass
    
    class hasName(DataProperty):
        range = [str]
    
    class albumName(hasName): domain = [Album]

    class artistName(hasName): domain = [Artist]

    class albumID(DataProperty, FunctionalProperty):
        domain = [Album]
        range  = [str] 
    
    class followers(DataProperty, FunctionalProperty):
        domain = [Artist]
        range = [int] 


In [11]:
with onto:
    class collaboratesWith(ObjectProperty, SymmetricProperty):
        domain = [Artist]
        range  = [Artist]

    class authorOf(Artist >> Song):
        inverse_property = writtenBy

    class hasFan(Artist >> Person): pass
    
    class isFanOf(Person >> Artist):
        inverse_property = hasFan

    Song.is_a.append(writtenBy.min(1, Artist))

    class CollaboratingArtist(Artist):
        equivalent_to = [Artist & collaboratesWith.some(Artist)]


In [12]:
#create instances of the classes
sza = Artist("SZA", namespace=onto)
sos = Album('SOS', namespace=onto)
kill_bill = Song("kill_bill", namespace=onto)
miss_lauren_hill = Artist("miss_lauren_hill", namespace=onto)
drake = SoloArtist("Drake", namespace=onto)

sos.hasArtist = [sza]
kill_bill.writtenBy = [sza]
kill_bill.isOfAlbum = [sos]
sza.collaboratesWith = [miss_lauren_hill, drake]

fie = Person('Fie', namespace=onto)
fie.isFanOf.append(sza)

In [13]:
#write simple queries HINT (onto.search, ..)
print(sza.iri)
print(f'SOS has artist {sos.hasArtist}')
print(f'who collabs with someone? {onto.search(type = onto.Artist, collaboratesWith = '*')}')
print(f'Who is a fan of sza? {onto.search(type=onto.Person, isFanOf='*')}')

http://example.org/music_ontology.owl#SZA
SOS has artist [music_ontology.SZA]
who collabs with someone? [music_ontology.SZA, music_ontology.miss_lauren_hill, music_ontology.Drake]
Who is a fan of sza? [music_ontology.Fie]


In [14]:
#save
onto.save(file="data/my_music_ontology.owl", format='rdfxml')

## OWL reasoning 

Let's look at how reasoning works.

Owlready automatically gets the results of the reasoning from HermiT (a type of reasoner) and reclassifies Individuals and Classes. 

**Exercise 2**
1. think about which things are inferred from your OWL semantics. Query/look at your graph: do you see what you expected?
2. looking at the following tutorial [owlready2-reasoning](https://owlready2.readthedocs.io/en/latest/reasoning.html), which things have not yet been inferred? Run the owlready2 reasoner to:
    - infer these new triples
    - check your ontology and statements (individuals + attributes) for consistency
3. save your asserted + inferred triples to a new file 

In [15]:
#SZA, Travis or Drake was never inferred to be a person, only me
for p in Person.instances():
    print(p)

#Only Drake was declared to be a solo artist 
for s in SoloArtist.instances():
    print(s)

#SZA is inferred to be authorOf the song kill_bill, inverse of writtenBy
print(f'SZA is the author of {list(sza.authorOf)}')


music_ontology.Fie
music_ontology.Drake
SZA is the author of [music_ontology.kill_bill]


In [16]:
#no one is inferred to be a CollaboratingArtist
for p in CollaboratingArtist.instances():
    print(f'Collaborating artists: {p}')


In [17]:
print("SZA collaboratesWith:", list(sza.collaboratesWith))
print("Drake artist type:", drake.is_a)
print("Miss Lauren Hill artist type:", miss_lauren_hill.is_a)


SZA collaboratesWith: [music_ontology.miss_lauren_hill, music_ontology.Drake]
Drake artist type: [music_ontology.SoloArtist]
Miss Lauren Hill artist type: [music_ontology.Artist]


In [18]:
with onto: 
    sync_reasoner(infer_property_values=True)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /opt/anaconda3/envs/knowledge_graphs/lib/python3.14/site-packages/owlready2/hermit:/opt/anaconda3/envs/knowledge_graphs/lib/python3.14/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/v7/4rvhk5951gqck0hx9gq2y3040000gn/T/tmph0hk6zzy -Y
* Owlready2 * HermiT took 2.9291341304779053 seconds
* Owlready * Reparenting music_ontology.miss_lauren_hill: {music_ontology.Artist} => {music_ontology.CollaboratingArtist}
* Owlready * Reparenting music_ontology.SZA: {music_ontology.Artist} => {music_ontology.CollaboratingArtist}
* Owlready * Reparenting music_ontology.Drake: {music_ontology.SoloArtist} => {music_ontology.SoloArtist, music_ontology.CollaboratingArtist}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [19]:
print("SZA collaboratesWith:", list(sza.collaboratesWith))
print("Drake artist type:", drake.is_a)
print("Miss Lauren Hill artist type:", miss_lauren_hill.is_a)


SZA collaboratesWith: [music_ontology.miss_lauren_hill, music_ontology.Drake]
Drake artist type: [music_ontology.SoloArtist, music_ontology.CollaboratingArtist]
Miss Lauren Hill artist type: [music_ontology.CollaboratingArtist]


In [22]:
for p in CollaboratingArtist.instances():
    print(f'Collaborating artists: {p}')


Collaborating artists: music_ontology.SZA
Collaborating artists: music_ontology.miss_lauren_hill
Collaborating artists: music_ontology.Drake


In [21]:
onto.save(file='my_music_ontology_inferred.owl', format='rdfxml')

#### Querying inferred triples

**Exercise 3**
Query your inferred triples: 

- *.get_parents_of(entity)* accepts any entity (Class, property or individual), and returns the superclasses (for a class), the superproperties (for a property), or the classes (for an individual). 

- *.get_instances_of(Class)* returns the individuals that are asserted as belonging to the given Class in the ontology. (NB for obtaining all instances, independently of the ontology they are asserted in, use Class.instances()).

- *.get_children_of(entity)* returns the subclasses (or subproperties) that are asserted for the given Class or property in the ontology. (NB for obtaining all children, independently of the ontology they are asserted in, use entity.subclasses()).

In [28]:
print(f'Superclass of SoloArtist: {onto.get_parents_of(SoloArtist)}')
print(f'Subproperty of writtenBy: {onto.get_parents_of(writtenBy)}')
print(f'Individual Drake: {onto.get_parents_of(drake)}')

Superclass of SoloArtist: [music_ontology.Artist]
Subproperty of writtenBy: [music_ontology.createdBy]
Individual Drake: [music_ontology.SoloArtist, music_ontology.CollaboratingArtist]


In [30]:
print(f'Instances of SoloArtist: {onto.get_instances_of(SoloArtist)}')
print(f'Instances of Artist: {onto.get_instances_of(Artist)}')

Instances of SoloArtist: [music_ontology.Drake]
Instances of Artist: [music_ontology.SZA, music_ontology.miss_lauren_hill]


In [34]:
print(f'The subclasses of Artist: {onto.get_children_of(Artist)}')

The subclasses of Artist: [music_ontology.SoloArtist, music_ontology.ArtistType, music_ontology.CollaboratingArtist]


## Hybrid Intelligence ontology

**Exercise 4**
We can use owlready2 to work with the Hybrid Intelligence (HI) ontology which you will be using for your own project. Using the tools from above, perform the following:
1. Load the HI ontology using owlready2 from the Data folder.
2. Create at least 1 new class with a class restriction.
3. Create at least 1 new object property with an object property restriction.
4. Create at least 1 new data property with a restricted domain and range.
